# 04.3 — Word2Vec CBOW

**CBOW (Continuous Bag of Words):** The inverse of skip-gram. Given the context words, predict the center word.

**Skip-gram vs CBOW:**
- Skip-gram: better for rare words (predicts many contexts per word)
- CBOW: faster, slightly better for frequent words (averages context signal)
- Both learn similar embeddings — skip-gram is more popular in practice

---

In [ ]:
import numpy as np
import re, random
from collections import Counter

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())
def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class Word2VecCBOW:
    """
    CBOW: predict center word from average of context embeddings.
    
    Forward pass:
    1. Look up context word embeddings in W_in
    2. Average them -> context vector h
    3. Dot with all output vectors: scores = W_out @ h
    4. Softmax -> probability distribution over vocab
    
    With negative sampling:
    - No softmax needed
    - Binary classification: is this a real (context->center) pair?
    """
    
    def __init__(self, embed_dim=50, window=2, n_negative=5, lr=0.025, n_epochs=5):
        self.embed_dim = embed_dim
        self.window = window
        self.n_negative = n_negative
        self.lr = lr
        self.n_epochs = n_epochs
        self.vocab = []
        self.word2idx = {}
        self.W_in = self.W_out = self.noise_dist = None
    
    def _build_vocab(self, corpus):
        counts = Counter(w for sent in corpus for w in sent)
        self.vocab = sorted(counts.keys())
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        freqs = np.array([counts[w] for w in self.vocab], dtype=float) ** 0.75
        self.noise_dist = freqs / freqs.sum()
        return counts
    
    def fit(self, corpus: list[str]):
        tokenized = [tokenize(s) for s in corpus]
        counts = self._build_vocab(tokenized)
        V = len(self.vocab)
        
        self.W_in  = (np.random.rand(V, self.embed_dim) - 0.5) / self.embed_dim
        self.W_out = np.zeros((V, self.embed_dim))
        
        for epoch in range(self.n_epochs):
            total_loss = 0.0; n = 0
            for sent in tokenized:
                tokens = [w for w in sent if w in self.word2idx]
                for i, center in enumerate(tokens):
                    c_idx = self.word2idx[center]
                    win = random.randint(1, self.window)
                    ctx_words = tokens[max(0,i-win):i] + tokens[i+1:i+win+1]
                    if not ctx_words: continue
                    
                    ctx_indices = [self.word2idx[w] for w in ctx_words]
                    
                    # Average context vectors (h)
                    h = self.W_in[ctx_indices].mean(axis=0)
                    
                    # Positive: predict center from context
                    u_center = self.W_out[c_idx]
                    pos_score = np.dot(h, u_center)
                    pos_loss = -np.log(sigmoid(pos_score) + 1e-10)
                    
                    # Gradient for center output vector
                    grad_u = (sigmoid(pos_score) - 1) * h
                    self.W_out[c_idx] -= self.lr * grad_u
                    
                    # Gradient flowing back to input embeddings
                    grad_h = (sigmoid(pos_score) - 1) * u_center
                    
                    # Negative samples
                    neg_loss = 0.0
                    for neg_idx in np.random.choice(V, size=self.n_negative, p=self.noise_dist):
                        if neg_idx == c_idx: continue
                        u_neg = self.W_out[neg_idx]
                        neg_score = np.dot(h, u_neg)
                        neg_loss += -np.log(sigmoid(-neg_score) + 1e-10)
                        self.W_out[neg_idx] -= self.lr * sigmoid(neg_score) * h
                        grad_h += sigmoid(neg_score) * u_neg
                    
                    # Distribute gradient equally to all context words
                    grad_per_ctx = grad_h / len(ctx_indices)
                    for ctx_idx in ctx_indices:
                        self.W_in[ctx_idx] -= self.lr * grad_per_ctx
                    
                    total_loss += pos_loss + neg_loss
                    n += 1
            
            self.lr *= 0.9
            print(f"  Epoch {epoch+1}: loss={total_loss/max(n,1):.4f}")
        return self
    
    def get_vector(self, word):
        if word not in self.word2idx: return np.zeros(self.embed_dim)
        return self.W_in[self.word2idx[word]]
    
    def most_similar(self, word, n=5):
        vec = self.get_vector(word)
        norm = np.linalg.norm(vec)
        if norm == 0: return []
        norms = np.linalg.norm(self.W_in, axis=1, keepdims=True)
        norms[norms==0] = 1
        sims = (self.W_in / norms) @ (vec / norm)
        top = sims.argsort()[::-1][1:n+1]
        return [(self.vocab[i], float(sims[i])) for i in top]

corpus = [
    "the king and queen rule the kingdom together",
    "the man walked to the store and the woman stayed",
    "dogs and cats are common household pets",
    "machine learning and deep learning use neural networks",
    "paris is the capital and london is also a capital city",
    "france and england are countries in europe",
] * 30

cbow = Word2VecCBOW(embed_dim=30, window=3, n_epochs=10, lr=0.05)
cbow.fit(corpus)

print("\nCBOW embeddings:")
for word in ['king', 'cat', 'learning', 'france']:
    if word in cbow.word2idx:
        print(f"  {word}: {cbow.most_similar(word, 4)}")

In [ ]:
# Key difference: averaging context vectors
# CBOW treats context as a bag (order doesn't matter)
# This is why 'bag' is in the name

# When does order matter?
pairs = [
    ("dog bites man", "man bites dog"),        # different meanings
    ("I saw her duck", "she saw the duck"),    # ambiguous
    ("not good", "good not"),                  # negation
]
print("CBOW limitation: order-insensitive")
print("These pairs have the same CBOW context vectors (just different center):")
for a, b in pairs:
    a_toks, b_toks = tokenize(a), tokenize(b)
    a_set, b_set = set(a_toks), set(b_toks)
    same_vocab = a_set == b_set
    print(f"  {a!r:30} vs {b!r:30}  same vocab={same_vocab}")

print()
print("Skip-gram doesn't fully solve this either — it's also bag-of-contexts.")
print("Order-sensitivity requires architectures with position encoding (Transformer).")

## Summary

**CBOW vs Skip-gram:**

| | Skip-gram | CBOW |
|-|-----------|------|
| Objective | center → context | context → center |
| Good for | rare words | frequent words |
| Training speed | slower | faster (avg gradient) |
| Accuracy | slightly better | slightly worse |

**Both share the same fundamental limitation:** one static vector per word. 'bank' gets one vector no matter the context.

---
**Next:** 04.4 — GloVe